In [1]:
# pip install -U pandas pyarrow

In [3]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm  # barra de progresso opcional

In [ ]:
# Caminho da pasta onde está o notebook (pasta atual)
pasta = Path(".")  # ou Path("/Users/seuusuario/caminho/completo") se quiser fixar

# Filtrar apenas os arquivos do padrão desejado
arquivos = sorted(pasta.glob("RQUAL_8ind-*.xlsx"))

print(f"📂 Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(" -", a.name)

# Ler e concatenar todos
dfs = []
for arq in tqdm(arquivos, desc="📖 Lendo planilhas RQUAL"):
    df = pd.read_excel(arq, engine="openpyxl")
    df["__arquivo_origem"] = arq.name  # rastreabilidade
    dfs.append(df)

# Unir tudo
base_rqual = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Base unificada: {len(base_rqual):,} linhas × {len(base_rqual.columns)} colunas")

# Verificação rápida
print("\n📊 Preview:")
display(base_rqual.head(3))

📂 Arquivos encontrados: 12
 - RQUAL_8ind-A.xlsx
 - RQUAL_8ind-BA.xlsx
 - RQUAL_8ind-CD.xlsx
 - RQUAL_8ind-E-G-MA.xlsx
 - RQUAL_8ind-MG.xlsx
 - RQUAL_8ind-MT-MS.xlsx
 - RQUAL_8ind-P.xlsx
 - RQUAL_8ind-RJ.xlsx
 - RQUAL_8ind-RN-RO-RR.xlsx
 - RQUAL_8ind-RS.xlsx
 - RQUAL_8ind-SC-SE-TO.xlsx
 - RQUAL_8ind-SP.xlsx


📖 Lendo planilhas RQUAL:  75%|█████████████▌    | 9/12 [05:12<01:31, 30.61s/it]

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm  # barra de progresso opcional

# Caminho dos arquivos
pasta = Path(".")
arquivos = sorted(pasta.glob("*.xlsx"))

print(f"📂 {len(arquivos)} arquivos encontrados (total ~500MB)\n")

# Pasta temporária para Parquet
tmp_dir = pasta / "tmp_parquet"
tmp_dir.mkdir(exist_ok=True)

# 1️⃣ Converter cada Excel em Parquet incrementalmente
for arq in tqdm(arquivos, desc="Convertendo para Parquet"):
    nome = arq.stem
    out = tmp_dir / f"{nome}.parquet"
    if not out.exists():
        df = pd.read_excel(arq, engine="pyarrow")
        df.to_parquet(out, index=False)
    else:
        print(f"↪️ {nome}.parquet já existe, pulando...")

# 2️⃣ Concatenar todos os Parquet
parquets = sorted(tmp_dir.glob("*.parquet"))
dfs = [pd.read_parquet(p) for p in tqdm(parquets, desc="Lendo Parquet")]
base = pd.concat(dfs, ignore_index=True)

# 3️⃣ Salvar base final
saida = pasta / "base_unificada.parquet"
base.to_parquet(saida, index=False)

print(f"\n✅ Base final criada: {saida.name}")
print(f"→ Linhas: {len(base):,} | Colunas: {len(base.columns)}")
print(f"→ Tamanho em disco (Parquet): ~{saida.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Salvar em Parquet — para abrir depois em segundos
base_rqual.to_parquet("base_RQUAL_unificada.parquet", index=False)